In [1]:
from pathlib import Path
import os

from import_libraries import *  # NOSONAR
from global_variables import *  # NOSONAR
import common_libraries.io_library as io_l

from ml_classification.ml_classification_pipeline import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\xicor\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
WORK_PATH = Path("C:/work_dir/k_makedonia-2023-2024")

PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
OUTPUT_PATH = WORK_PATH / "3_ml_classification"

class_column = "initial_label_code"
parcel_id_column = "parcel_code"


In [3]:
gdf_parcels = io_l.read_data(str(PARCELS_PATH))
gdf_parcels.head(3)

Reading file: C:/work_dir/k_makedonia-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet


parcel_code  initial_label_code  initial_label  predicted_label_code  \
0   174459115                 129         fallow                  110    
1   277728242                 119  grass_fodders                  111    
2   277728254                 110          wheat                  110    

  predicted_label  prediction_probability set_type             geometry  \
0           wheat         0.3402073400        test  POLYGON ((393737...   
1          barley         0.3240505500        test  POLYGON ((398487...   
2           wheat         0.6185110000       train  POLYGON ((394917...   

   batch_number  eligible_pixel_count  meets_minimum_pixel_count  \
0             2                   31                  True         
1             2                  107                  True         
2             2                   53                  True         

    parcel_area_m2  approx_pixel_count  B02_bowley_skewness__20231001  \
0  3211.2943605089       32.1129436051         0.1897437230             
1 10734.0982834638      107.3409828346        -0.0424528435             
2  5151.4461354561       51.5144613546         0.0909087391             

   B02_bowley_skewness__20231101  B02_bowley_skewness__20231201  \
0         0.6631583570                  -0.1100483040             
1         0.1415104153                  -0.2620543240             
2        -0.0461534935                   0.1688318474             

   B02_bowley_skewness__20240101  B02_bowley_skewness__20240201  \
0        -0.3333324135                  -0.0072988493             
1        -0.1783789792                   0.1836735004             
2         0.1764707601                   0.2413794875             

   B02_bowley_skewness__20240301  B02_bowley_skewness__20240401  \
0         0.2421651091                   0.0945952069             
1         0.0222209713                   0.1718064739             
2         0.3142858765                  -0.0392156147             

   B02_bowley_skewness__20240501  B02_bowley_skewness__20240601  \
0         0.4154946870                  -0.0569103229             
1         0.2258078471                   0.2760726774             
2        -0.1500006519                  -0.2592597703             

   B02_bowley_skewness__20240701  B02_bowley_skewness__20240801  \
0         0.0823524874                  -0.0574722487             
1         0.1181087950                   0.2250000466             
2         0.3846162663                  -0.0000017739             

   B02_bowley_skewness__20240901  B02_coefficient_of_variation__20231001  \
0         0.4387098480                   0.1465092263                      
1         0.1176474133                   0.1441161505                      
2        -0.4909085983                   0.1402357804                      

   B02_coefficient_of_variation__20231101  \
0         0.1118483181                      
1         0.1578468645                      
2         0.1619581420                      

   B02_coefficient_of_variation__20231201  \
0         0.1197034993                      
1         0.1057216984                      
2         0.0970404892                      

   B02_coefficient_of_variation__20240101  \
0         0.0855427087                      
1         0.1221690341                      
2         0.1108020677                      

   B02_coefficient_of_variation__20240201  \
0         0.1786684286                      
1         0.1437596861                      
2         0.1400282341                      

   B02_coefficient_of_variation__20240301  \
0         0.3276227450                      
1         0.2055736850                      
2         0.1723590942                      

   B02_coefficient_of_variation__20240401  \
0         0.3049161501                      
1         0.2057088070                      
2         0.1078614313                      

   B02_coefficient_of_variation__20240501  \
0         0.1922265116                      
1         0.

In [4]:
label_counts = gdf_parcels[class_column].value_counts()
rare_labels = label_counts[label_counts < 100].index

gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = 9999
print(gdf_parcels[class_column].value_counts())

initial_label_code
110     49902
119     41226
111     12418
129      5786
112      4763
109      3939
114      3932
116      3040
113      2781
101      1398
128      1185
102       765
108       644
117       598
130       552
9999      295
106       289
121       236
127       234
126       118
125       102
Name: count, dtype: int64


In [5]:
gdf_parcels = gdf_parcels.to_crs("EPSG:2100")
gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

base_cols = [parcel_id_column, class_column, "geometry"]
extra_features = ["elevation_mean_m", "distance_to_nearest_lake_m", "GAIOIKANOT", "x", "y"]
temporal_features = ("_median", "_sd", "_range",)
original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in temporal_features)]

gdf_parcels = gdf_parcels[base_cols + original_features + extra_features].copy()
features = original_features + extra_features

gdf_parcels = gdf_parcels[gdf_parcels[class_column] != 9999].copy()
print(f"Final features for modeling ({len(features)} features): {features}")
display(gdf_parcels.head(3))

Final features for modeling (545 features): ['B02_median__20231001', 'B02_median__20231101', 'B02_median__20231201', 'B02_median__20240101', 'B02_median__20240201', 'B02_median__20240301', 'B02_median__20240401', 'B02_median__20240501', 'B02_median__20240601', 'B02_median__20240701', 'B02_median__20240801', 'B02_median__20240901', 'B02_range__20231001', 'B02_range__20231101', 'B02_range__20231201', 'B02_range__20240101', 'B02_range__20240201', 'B02_range__20240301', 'B02_range__20240401', 'B02_range__20240501', 'B02_range__20240601', 'B02_range__20240701', 'B02_range__20240801', 'B02_range__20240901', 'B02_sd__20231001', 'B02_sd__20231101', 'B02_sd__20231201', 'B02_sd__20240101', 'B02_sd__20240201', 'B02_sd__20240301', 'B02_sd__20240401', 'B02_sd__20240501', 'B02_sd__20240601', 'B02_sd__20240701', 'B02_sd__20240801', 'B02_sd__20240901', 'B03_median__20231001', 'B03_median__20231101', 'B03_median__20231201', 'B03_median__20240101', 'B03_median__20240201', 'B03_median__20240301', 'B03_me

,parcel_code,initial_label_code,geometry,B02_median__20231001,B02_median__20231101,B02_median__20231201,B02_median__20240101,B02_median__20240201,B02_median__20240301,B02_median__20240401,B02_median__20240501,B02_median__20240601,B02_median__20240701,B02_median__20240801,B02_median__20240901,B02_range__20231001,B02_range__20231101,B02_range__20231201,B02_range__20240101,B02_range__20240201,B02_range__20240301,B02_range__20240401,B02_range__20240501,B02_range__20240601,B02_range__20240701,B02_range__20240801,B02_range__20240901,B02_sd__20231001,B02_sd__20231101,B02_sd__20231201,B02_sd__20240101,B02_sd__20240201,B02_sd__20240301,B02_sd__20240401,B02_sd__20240501,B02_sd__20240601,B02_sd__20240701,B02_sd__20240801,B02_sd__20240901,B03_median__20231001,B03_median__20231101,B03_median__20231201,B03_median__20240101,B03_median__20240201,B03_median__20240301,B03_median__20240401,B03_median__20240501,B03_median__20240601,B03_median__20240701,B03_median__20240801,B03_median__20240901,B03_range__20231001,B03_range__20231101,B03_range__20231201,B03_range__20240101,B03_range__20240201,B03_range__20240301,B03_range__20240401,B03_range__20240501,B03_range__20240601,B03_range__20240701,B03_range__20240801,B03_range__20240901,B03_sd__20231001,B03_sd__20231101,B03_sd__20231201,B03_sd__20240101,B03_sd__20240201,B03_sd__20240301,B03_sd__20240401,B03_sd__20240501,B03_sd__20240601,B03_sd__20240701,B03_sd__20240801,B03_sd__20240901,B04_median__20231001,B04_median__20231101,B04_median__20231201,B04_median__20240101,B04_median__20240201,B04_median__20240301,B04_median__20240401,B04_median__20240501,B04_median__20240601,B04_median__20240701,B04_median__20240801,B04_median__20240901,B04_range__20231001,B04_range__20231101,B04_range__20231201,B04_range__20240101,B04_range__20240201,B04_range__20240301,B04_range__20240401,B04_range__20240501,B04_range__20240601,B04_range__20240701,B04_range__20240801,B04_range__20240901,B04_sd__20231001,B04_sd__20231101,B04_sd__20231201,B04_sd__20240101,B04_sd__20240201,B04_sd__20240301,B04_sd__20240401,B04_sd__20240501,B04_sd__20240601,B04_sd__20240701,B04_sd__20240801,B04_sd__20240901,B05_median__20231001,B05_median__20231101,B05_median__20231201,B05_median__20240101,B05_median__20240201,B05_median__20240301,B05_median__20240401,B05_median__20240501,B05_median__20240601,B05_median__20240701,B05_median__20240801,B05_median__20240901,B05_range__20231001,B05_range__20231101,B05_range__20231201,B05_range__20240101,B05_range__20240201,B05_range__20240301,B05_range__20240401,B05_range__20240501,B05_range__20240601,B05_range__20240701,B05_range__20240801,B05_range__20240901,B05_sd__20231001,B05_sd__20231101,B05_sd__20231201,B05_sd__20240101,B05_sd__20240201,B05_sd__20240301,B05_sd__20240401,B05_sd__20240501,B05_sd__20240601,B05_sd__20240701,B05_sd__20240801,B05_sd__20240901,B08_median__20231001,B08_median__20231101,B08_median__20231201,B08_median__20240101,B08_median__20240201,B08_median__20240301,B08_median__20240401,B08_median__20240501,B08_median__20240601,B08_median__20240701,B08_median__20240801,B08_median__20240901,B08_range__20231001,B08_range__20231101,B08_range__20231201,B08_range__20240101,B08_range__20240201,B08_range__20240301,B08_range__20240401,B08_range__20240501,B08_range__20240601,B08_range__20240701,B08_range__20240801,B08_range__20240901,B08_sd__20231001,B08_sd__20231101,B08_sd__20231201,B08_sd__20240101,B08_sd__20240201,B08_sd__20240301,B08_sd__20240401,B08_sd__20240501,B08_sd__20240601,B08_sd__20240701,B08_sd__20240801,B08_sd__20240901,B11_median__20231001,B11_median__20231101,B11_median__20231201,B11_median__20240101,B11_median__20240201,B11_median__20240301,B11_median__20240401,B11_median__20240501,B11_median__20240601,B11_median__20240701,B11_median__20240801,B11_median__20240901,B11_range__20231001,B11_range__20231101,B11_range__20231201,B11_range__20240101,B11_range__20240201,B11_range__20240301,B11_range__20240401,B11_range__20240501,B11_range__20240601,B11_range__20240701,B11_range__20240801,B11_ran

In [6]:
config = ClassificationPipelineConfig(
    project_dir=OUTPUT_PATH,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,

    apply_iqr=False,
    spatial_interpolation_method=None,

    cv_folds=5,
    test_size=0.2,
    spatial_split=True,
    spatial_split_grid_size=10,
    spatial_split_method="by_row",

    random_state=SEED_NUMBER,
    # n_jobs=1,

    scoring_primary="f1_macro",
    cv_ranking_method="score_minus_std",
    selection_type="soft_voting",
    top_voting_models=5,
    max_search_candidates=40,
    prediction_confidence_threshold=0.60,

    interpretability_top_models=5,
    interpretability_include_shap=False,

    optimize_class_probabilities=True,
    probability_multiplier_grid=(0.8, 1.0, 1.2, 1.5, 2.0),
    probability_optimization_iterations=2,
    probability_optimization_max_accuracy_drop=0.02,

    rank_confidence_enabled=True,
    force_retrain_models=False,

    label_balancing_method="random_oversample",
    lstm_temporal_statistics=temporal_features,
    selected_models=(
        "bagging",
        # "bayesian",
        # "decision_tree",
        "extra_trees",
        "random_forest",
        "gradient_boosting",
        # "lightgbm",
        "xgboost",
        # "knn",
        "neural_network",
        # "tensorflow_neural_network",
        # "keras_lstm",
   ),
    max_map_geometries=20000,
    open_html_report=True,
)

pipeline = GeospatialClassificationPipeline(config)


In [ ]:
summary_check = pipeline.run_check(df=gdf_parcels)
summary_check

In [ ]:
summary_prepare = pipeline.run_prepare()
summary_prepare

In [ ]:
summary_train = pipeline.run_train()
summary_train

In [7]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate


Function <run_evaluate> started on 2026-08-23 21:10:06

Evaluating models...
Reading file: C:/work_dir/k_makedonia-2023-2024/3_ml_classification/02_prepare/label_mapping.csv
Reading file: C:/work_dir/k_makedonia-2023-2024/3_ml_classification/03_train/training_summary.csv
Removing existing file: C:/work_dir/k_makedonia-2023-2024/3_ml_classification/04_evaluate/reports/random_forest_classification_report.csv
Writing file: C:/work_dir/k_makedonia-2023-2024/3_ml_classification/04_evaluate/reports/random_forest_classification_report.csv
Removing existing file: C:/work_dir/k_makedonia-2023-2024/3_ml_classification/04_evaluate/reports/extra_trees_classification_report.csv
Writing file: C:/work_dir/k_makedonia-2023-2024/3_ml_classification/04_evaluate/reports/extra_trees_classification_report.csv
Removing existing file: C:/work_dir/k_makedonia-2023-2024/3_ml_classification/04_evaluate/reports/gradient_boosting_classification_report.csv
Writing file: C:/work_dir/k_makedonia-2023-2024/3_ml_clas

{'selection_type': 'soft_voting',
 'selected_models': ['xgboost',
  'bagging',
  'gradient_boosting',
  'extra_trees',
  'random_forest'],
 'selected_metric': 'cv_score_minus_std',
 'selected_score': 0.3729697057135727,
 'selection_source': 'cross_validation',
 'labels': ['101',
  '102',
  '106',
  '108',
  '109',
  '110',
  '111',
  '112',
  '113',
  '114',
  '116',
  '117',
  '119',
  '121',
  '125',
  '126',
  '127',
  '128',
  '129',
  '130'],
 'class_probability_multipliers': {'101': 0.8,
  '102': 0.8,
  '106': 0.8,
  '108': 0.8,
  '109': 1.2,
  '110': 1.2,
  '111': 1.0,
  '112': 1.2,
  '113': 0.8,
  '114': 0.8,
  '116': 0.8,
  '117': 0.8,
  '119': 2.0,
  '121': 1.0,
  '125': 1.5,
  '126': 1.5,
  '127': 0.8,
  '128': 1.0,
  '129': 1.5,
  '130': 1.2},
 'probability_optimization': {'baseline_oof_macro_f1': 0.4571828610760975,
  'optimized_oof_macro_f1': 0.49316379207208694,
  'baseline_oof_accuracy': 0.5832290947109012,
  'optimized_oof_accuracy': 0.6300057875772455},
 'rank_confide

In [ ]:
summary_predict = pipeline.run_predict()
summary_predict

In [ ]:
# Show only rows that were unknown during training
classified_gdf = pd.read_csv(os.path.join(str(OUTPUT_PATH), "04_evaluate", "final_predictions_preview.csv"))
classified_gdf[parcel_id_column] = classified_gdf[parcel_id_column].astype(str)
print(len(classified_gdf), "rows in classified_gdf")
display(classified_gdf.head())

In [ ]:
from ml_classification.ml_classification_pipeline.core.persistence import load_joblib
test_df = load_joblib(Path(f"{OUTPUT_PATH}/02_prepare/test_dataset.joblib"))
test_df[parcel_id_column] = test_df[parcel_id_column].astype(str)
print(len(test_df), "rows in test_df")
test_df.head()

In [ ]:
test_ids =list(test_df[parcel_id_column].unique())
classified_gdf = classified_gdf[classified_gdf[parcel_id_column].isin(test_ids)]
print(len(classified_gdf), "rows in classified_gdf after filtering with test_ids")
classified_gdf["accuracy"] = classified_gdf[class_column] == classified_gdf[f"{class_column}_prediction"]
accuracy = classified_gdf["accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")